#Corrección de Taller Práctico 2 – Sistema RAG
#Autores:
#Alvaro José Cabrera
#Claudia Lorena Aragón Payán


# Fase 1: Selección de Componentes Clave del Sistema RAG

Para el sistema RAG de EcoMarket se seleccionó el modelo de embeddings:

sentence-transformers/all-MiniLM-L6-v2

Este modelo fue seleccionado debido a que ofrece un balance adecuado entre rendimiento, velocidad y bajo consumo computacional.

Los embeddings son fundamentales dentro de una arquitectura RAG porque convierten los documentos y las preguntas de los usuarios en vectores numéricos, permitiendo realizar búsquedas semánticas por similitud.
# Justificacion del modelo all-MiniLM-L6-v2 fue seleccionado considerando los siguientes factores:

Rendimiento

*  Genera embeddings semánticos de buena calidad.
Permite recuperar información relevante de manera eficiente.
Funciona correctamente para tareas de preguntas y respuestas.
Costo.


Costo:

*  Es completamente gratuito.
Puede ejecutarse localmente.
No requiere pago por API.
Recursos Computacionales
Es un modelo liviano.
Puede ejecutarse en CPU.
Tiene bajo consumo de memoria RAM.
Facilidad de Implementación
Tiene integración directa con LangChain.
Compatible con ChromaDB.
Fácil de usar en Google Colab.
Rendimiento para Español

Aunque el modelo no está específicamente entrenado para español, presenta un desempeño aceptable para tareas semánticas simples y prototipos académicos.

| Modelo               | Ventajas                         | Desventajas                |
| -------------------- | -------------------------------- | -------------------------- |
| all-MiniLM-L6-v2     | Ligero, rápido y gratuito        | Menor precisión en español |
| multilingual-e5-base | Mejor soporte multilingüe        | Más pesado                 |
| OpenAI Embeddings    | Alta precisión                   | Tiene costo                |
| BAAI/bge-small       | Excelente recuperación semántica | Mayor consumo de recursos  |


Impacto de los Embeddings en el Sistema RAG

El modelo de embeddings impacta directamente: la precisión de recuperación,
la calidad del contexto, la velocidad de búsqueda y la calidad final de las respuestas generadas.

Si el embedding recupera información incorrecta, el modelo generativo puede producir respuestas erróneas o alucinaciones.

Por esta razón, el embedding es uno de los componentes más importantes de un sistema RAG.

* Base de datos vectorial seleccionada para almacenar los embeddings se
ChromaDB debido a que es :
Es gratuita y open source.
Tiene integración sencilla con LangChain.
Permite almacenamiento local.
Facilita búsquedas por similitud.
Tiene bajo costo computacional.
Es adecuada para proyectos pequeños y medianos.
Comparación con Otras Bases Vectoriales
Base Vectorial	Ventajas	Desventajas
ChromaDB	Gratuita y fácil de usar	Menor escalabilidad
Pinecone	Muy escalable	Tiene costo
Weaviate	Flexible y potente	Mayor complejidad
FAISS	Muy rápida	Menor persistencia
Impacto de la Base Vectorial en el Sistema RAG

* La base vectorial influye directamente en:

velocidad de recuperación,
tiempo de respuesta,
escalabilidad,
almacenamiento de embeddings,
y precisión de búsqueda.

Una base vectorial eficiente permite recuperar información relevante rápidamente y mejorar la calidad de las respuestas del sistema.

Conclusión de la La combinación de: all-MiniLM-L6-v2 y ChromaDB
permite construir un sistema RAG eficiente, económico y funcional para el caso EcoMarket.

Esta arquitectura es adecuada para prototipos académicos y sistemas de atención al cliente con un volumen moderado de documentos.

# Fase 2: Creación de la Base de Conocimiento de Documentos

Para construir la base de conocimiento se seleccionaron documentos relevantes para el servicio de atención al cliente de EcoMarket.

Los documentos utilizados fueron: Políticas de devolución, Información de envío,
Métodos de pago, Preguntas frecuentes y la Disponibilidad de productos.

* Estos documentos fueron seleccionados porque representan las consultas más frecuentes realizadas por los clientes de una empresa de e-comme.

* Estrategia de Segmentación (Chunking)

Se utilizó una estrategia de chunking mediante: RecursiveCharacterTextSplitter

* *La segmentación divide documentos grandes en fragmentos más pequeños llamados chunks.

Configuración Utilizada
chunk_size = 300
chunk_overlap = 50
Justificación del Chunking

La segmentación es importante porque:

Chunks muy pequeños pueden perder contexto.
Chunks muy grandes disminuyen precisión.
La segmentación mejora la recuperación semántica.
Facilita búsquedas más específicas.

Se utilizó overlap para conservar continuidad semántica entre fragmentos consecutivos.

Proceso de Indexación

El proceso de indexación consistió en:

Cargar documentos desde archivos .txt.
Dividir documentos en chunks.
Convertir chunks en embeddings.
Almacenar embeddings en ChromaDB.
Recuperar documentos usando similitud semántica.

El flujo implementado fue:

El usuario realiza una pregunta.
La pregunta se convierte en embedding.
ChromaDB busca los chunks más similares.
Se recupera el contexto relevante.
El modelo generativo genera la respuesta final.

A manera de conclución La combinación de: documentos relevantes,
chunking adecuado, embeddings, y almacenamiento vectorial permite mejorar significativamente la recuperación de información y reducir respuestas incorrectas o alucinaciones.

In [ ]:
############################################################
# TALLER PRÁCTICO #2
# SISTEMA RAG - ECOMARKET
############################################################
############################################################
# INSTALAR LIBRERÍAS
############################################################

!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-text-splitters
!pip install -q sentence-transformers
!pip install -q chromadb
!pip install -q transformers
!pip install -q pypdf

############################################################
# IMPORTAR LIBRERÍAS
############################################################

############################################################
# IMPORTAR LIBRERÍAS
############################################################
# Carga documentos desde archivos
from langchain_community.document_loaders import DirectoryLoader

# Divide documentos en chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Modelo de embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# Base de datos vectorial
from langchain_community.vectorstores import Chroma

# Modelo generativo
from transformers import pipeline

import os

In [ ]:
############################################################
# 2. IMPORTAR LIBRERÍAS
############################################################

from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_community.vectorstores import Chroma

from transformers import pipeline

import os

# Se crea una carpeta llamada docs donde se almacenarán los documentos utilizados como base de conocimiento

In [ ]:
############################################################
# 3. CARGAR DOCUMENTOS
############################################################

# Crear carpeta docs si no existe
os.makedirs("docs", exist_ok=True)

# Documento relacionado con devoluciones

In [ ]:
# DOCUMENTO 1
############################################################

with open("docs/politicas.txt", "w", encoding="utf-8") as f:
    f.write("""
EcoMarket permite devoluciones dentro de los primeros 30 días posteriores a la compra.
El producto debe conservar su empaque original y no presentar daños.
Los reembolsos tardan entre 5 y 7 días hábiles.
""")

In [ ]:

############################################################
# DOCUMENTO 2
############################################################

with open("docs/envios.txt", "w", encoding="utf-8") as f:
    f.write("""
Los envíos nacionales tardan entre 3 y 5 días hábiles.
Los envíos internacionales pueden tardar entre 10 y 15 días.
Los costos de envío dependen de la ubicación del cliente.
""")

# Documento relacionado con métodos de pago

In [ ]:
############################################################
# DOCUMENTO 3
############################################################

with open("docs/pagos.txt", "w", encoding="utf-8") as f:
    f.write("""
EcoMarket acepta pagos con tarjeta de crédito, débito y transferencia bancaria.
Las compras superiores a $200 tienen envío gratuito.
Las facturas se generan automáticamente después de cada compra.
""")

In [ ]:
!pip install -q unstructured

In [ ]:

############################################################
# CARGAR DOCUMENTOS DESDE LA CARPETA
############################################################

from langchain_community.document_loaders import TextLoader

documents = []

archivos = [
    "docs/politicas.txt",
    "docs/envios.txt",
    "docs/pagos.txt"
]

for archivo in archivos:
    loader = TextLoader(archivo, encoding="utf-8")
    documents.extend(loader.load())

print("Cantidad de documentos cargados:")
print(len(documents))

print("Cantidad de documentos cargados:")
print(len(documents))

# Este paso divide los documentos en chunks
# para mejorar la recuperación semántica

# El chunking divide documentos grandes en fragmentos pequeños llamados chunks
# Esto mejora: recuperación semántica, precisión y la búsqueda contextual

In [ ]:

############################################################
# 4. CHUNKING
############################################################

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300,
    chunk_overlap = 50
)

chunks = text_splitter.split_documents(documents)

print("\nCantidad de chunks generados:")
print(len(chunks))

# Se utiliza all-MiniLM-L6-v2 porque es ligero, rápido y gratuito
# Los embeddings convierten texto en vectores numéricos

In [ ]:
############################################################
# 5. MODELO DE EMBEDDINGS
############################################################

embedding_model = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

# ChromaDB almacena los embeddings permitiendo realizar búsquedas por similitud


In [ ]:

############################################################
# 6. BASE DE DATOS VECTORIAL (CHROMADB)
############################################################

vectordb = Chroma.from_documents(
    documents = chunks,
    embedding = embedding_model,
    persist_directory = "chroma_db"
)


# El retriever busca los chunks más similares a la pregunta del usuario

In [ ]:
############################################################
# 7. CREAR RETRIEVER
############################################################

retriever = vectordb.as_retriever(
    search_kwargs = {"k": 2}
)


In [ ]:
!pip install -U transformers

# El modelo generativo crea la respuesta final usando el contexto recuperado

In [ ]:
generator = pipeline(
    "text-generation",
    model = "tiiuae/falcon-rw-1b"
)

In [ ]:
############################################################
# 9. PREGUNTA DEL USUARIO
############################################################

pregunta = "¿Cuántos días tengo para devolver un producto?"

# Se buscan los documentos más relevantes utilizando similitud semántica

In [ ]:
############################################################
# 10. RECUPERAR CONTEXTO
############################################################

docs_recuperados = retriever.invoke(pregunta)

contexto = "\n".join(
    [doc.page_content for doc in docs_recuperados]
)

print("\nContexto recuperado:\n")

print(contexto)

# El prompt combina: contexto recuperado y pregunta del usuario

In [ ]:
############################################################
# 11. CREAR PROMPT
############################################################

prompt = f"""
Responde la pregunta utilizando únicamente el contexto proporcionado.

Contexto:
{contexto}

Pregunta:
{pregunta}

Respuesta:
"""


# El modelo genera una respuesta basada en el contexto recuperado

In [ ]:
############################################################
# 12. GENERAR RESPUESTA
############################################################

respuesta = generator(
    prompt,
    max_length = 100
)


In [ ]:
############################################################
# 13. MOSTRAR RESPUESTA FINAL
############################################################

print("\nRespuesta generada:\n")

print(respuesta[0]["generated_text"])

In [ ]:

############################################################
# 14. EXPLICACIÓN FINAL
############################################################

print("\nFLUJO DEL SISTEMA RAG:")
print("1. Se cargan documentos reales")
print("2. Se dividen en chunks")
print("3. Se generan embeddings")
print("4. Los embeddings se guardan en ChromaDB")
print("5. Se recupera el contexto más relevante")
print("6. FLAN-T5 genera la respuesta final")

#Conclusión

La implementación permitió comprender cómo interactúan:

embeddings,
chunking,
retrieval,
bases vectoriales,
y modelos generativos

dentro de una arquitectura RAG.

Además, el sistema logró recuperar información relevante desde documentos internos y generar respuestas contextualizadas, reduciendo la probabilidad de alucinaciones frente a un modelo generativo tradicional sin recuperación de contexto.